Notebook to place a pattern on a FIB image

In [1]:
# general
from __future__ import annotations
import re
import tempfile
import typing
import yaml
import logging
from dataclasses import dataclass, field
from importlib import resources
from pathlib import Path
from os import PathLike

import tifffile
import matplotlib.pyplot as plt

# for the fibsem structures
from fibsem.milling import get_milling_stages

# Set up test microscope
from fibsem import utils
from fibsem.structures import Point, ImageSettings
from fibsem.milling.patterning.plotting import draw_milling_patterns

# Adaptive polishing
from adaptive_polish.strategy import BitmapAdaptivePolishMillingStrategy
from adaptive_polish._dataclasses import CycleInformation, CycleTimestamps

if typing.TYPE_CHECKING:
    from numpy.typing import NDArray

    from fibsem.milling import FibsemMillingStage
    from fibsem.structures import FibsemImage


class NoImageFound(FileNotFoundError):
    pass


class NoExperimentFound(NoImageFound):
    pass

Default configuration default-configuration. Configuration Path: c:\Users\tpr78264\code\adapive_polish_workspace\adaptive_polish\.venv\lib\site-packages\fibsem\config\microscope-configuration.yaml


c:\Users\tpr78264\code\adapive_polish_workspace\adaptive_polish\.venv\lib\site-packages\networkx\utils\backends.py:132: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends = _get_backends("networkx.plugins")
C:\Program Files\Enthought\Python\envs\AutoScript\Lib\site-packages\numexpr\interpreter.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import sys, pkg_resources, importlib.util


In [2]:
base_path = (
    Path.home()
    / "OneDrive - The Rosalind Franklin Institute"
    / "Documents"
    / "test data"
    / "adaptive milling"
)

model_path = (
    base_path / "sem_models" / "Gen1" / "gen01_V8_FPN_RGB" / "cryo_sem_epoch_69.pth"
)
model_generation = "1.4fpn"

experiment_path = (
    base_path
    / "AM session 20251305"
    / "20250512_AM_all"
    / "20250513_AM_ON"
    / "AutoLamella-2025-05-13-16-35"
)

plots_directory = experiment_path / "bitmap_pattern_plots"

# Path to the microscope configuration that will be used
microscope_config_path = (
    Path(str(resources.files("fibsem"))) / "config" / "microscope-configuration.yaml"
).resolve()

In [3]:
def setup_test_protocol(
    protocol_template: dict[str, typing.Any] | str | PathLike[str],
    temporary_directory: Path,
    model_path: str | PathLike[str],
    model_generation: str,
    **protocol_kwargs: typing.Any,
) -> Path:
    """Sets up a demo microscope"""
    if isinstance(protocol_template, (str, PathLike)):
        with Path(protocol_template).open() as f:
            protocol_template_dict = yaml.safe_load(f)
    else:
        protocol_template_dict = protocol_template

    # Rough 1 to bitmap
    rough_milling_stage = protocol_template_dict["milling"]["mill_rough"][0]
    rough_milling_stage["strategy"]["name"] = "BitmapAdaptivePolishing"
    rough_milling_stage["strategy"]["config"].update(
        {
            "model_path": str(model_path),
            "model_generation": model_generation,
            **protocol_kwargs,
        }
    )

    # AP to bitmap
    for i, polishing_stage in enumerate(
        protocol_template_dict["milling"]["mill_polishing"]
    ):
        if polishing_stage["strategy"]["name"] == "AdaptivePolishing":
            break
    polishing_stage["strategy"]["name"] = "BitmapAdaptivePolishing"
    polishing_stage["strategy"]["config"].update(
        {
            "model_path": str(model_path),
            "model_generation": model_generation,
            **protocol_kwargs,
        }
    )
    protocol_template_dict["milling"]["mill_polishing"][i] = polishing_stage

    protocol_path = temporary_directory / "protocol.yaml"
    with protocol_path.open("w") as f:
        yaml.safe_dump(protocol_template_dict, f)

    return protocol_path

In [4]:
def get_fib_and_sem_directories(
    data_directory: str | PathLike[str],
) -> tuple[Path, Path]:
    data_directory = Path(data_directory)
    if not data_directory.is_dir():
        raise FileNotFoundError(f"{data_directory} could not be found")
    fib_image_dir = data_directory / "fib"
    sem_image_dir = data_directory / "sem"
    if not fib_image_dir.is_dir():
        fib_image_dir = data_directory / "FIB"
        if not fib_image_dir.is_dir():
            raise FileNotFoundError("Failed to find FIB directory")
    if not sem_image_dir.is_dir():
        sem_image_dir = data_directory / "SEM"
        if not sem_image_dir.is_dir():
            raise FileNotFoundError("Failed to find SEM directory")
    return (fib_image_dir, sem_image_dir)

In [5]:
def run_placement(
    cycle: int,
    milling_stages: list[FibsemMillingStage],
    sem_image: FibsemImage,
    fib_image: FibsemImage,
    plots_directory: str | PathLike[str],
) -> None:
    updated_milling_stages: list[FibsemMillingStage] = []
    for i, milling_stage in enumerate(milling_stages):
        strategy = typing.cast(
            BitmapAdaptivePolishMillingStrategy, milling_stage.strategy
        )
        if strategy.model is None:
            strategy._load_model()
            logging.info("Loaded model")

        assert sem_image.metadata is not None

        logging.info("Getting lamella info")
        cycle_info = CycleInformation(
            milling_cycle=cycle,
            identifier=f"{sem_image.metadata.image_settings.filename}_{i}",
            timestamps=CycleTimestamps(),
        )
        lamella_info = strategy._get_lamella_info(
            cycle_info=cycle_info,
            fib_image=fib_image,
            sem_image=sem_image,
        )

        logging.info("Updating milling stage")
        # Make a copy of the milling stage before updating the pattern
        updated_milling_stage = strategy._update_milling_stage(
            stage=milling_stage, lamella_info=lamella_info
        )
        updated_milling_stages.append(updated_milling_stage)

        # TODO: Try using AnnotationBbox and OffsetImage for bitmap plotting (current method is too slow for large images)
        logging.info("Creating stage plot")
        strategy._create_milling_cycle_plot(
            plots_directory=Path(plots_directory),
            lamella_info=lamella_info,
            stage=updated_milling_stage,
        )

    logging.info("Creating multistage plot")
    fig, _ = draw_milling_patterns(
        fib_image,
        updated_milling_stages,
        crosshair=False,
        highlight_overlaps=True,
        ax=None,
    )
    figure_path = (
        Path(plots_directory)
        / f"{lamella_info.identifier}_{cycle}_bitmap_pattern_plot.tif"
    )
    print(f'Saving figure to "{figure_path}"')
    fig.savefig(figure_path)
    plt.close("all")


In [6]:
@dataclass
class Metadata:
    pixel_size: Point
    image_settings: ImageSettings


@dataclass
class TestImageInfo:
    path: Path
    data: NDArray[typing.Any]
    pixel_size_m: float
    metadata: Metadata = field(init=False)

    def __post_init__(self) -> None:
        self.metadata = Metadata(
            pixel_size=Point(x=self.pixel_size_m, y=self.pixel_size_m),
            image_settings=ImageSettings(
                hfw=4e-5,
                resolution=(self.data.shape[0], self.data.shape[1]),
                path=str(self.path.parents),
                filename=self.path.stem,
            ),
        )


In [7]:
def get_array_and_pixel_size(image_path: Path) -> tuple[NDArray[typing.Any], float]:
    with tifffile.TiffFile(image_path) as tif:
        image_array = tif.asarray()
        if tif.shaped_metadata is None:
            raise ValueError("Failed to read pixel size as shaped_metadata is None")
        return image_array, tif.shaped_metadata[0]["pixel_size"]["x"]

In [8]:
def process_lamella(
    fib_path: Path,
    sem_path: Path,
    plot_dir: Path,
    milling_stages: list[FibsemMillingStage],
) -> None:
    image_regex = re.compile(r"^([\w_\-\. ]+)_img_(\d{3})_(?:SEM|FIB)$", re.I)
    for i, sem_image_path in enumerate(sem_path.glob("*.tif"), 1):
        m = image_regex.match(sem_image_path.stem)
        if m is None:
            raise NoImageFound(f"'{sem_image_path.stem}' is not a valid image name")

        fib_image_path = fib_path / f"{m.group(1)}_img_{m.group(2)}_FIB.tif"
        sem_info = TestImageInfo(
            sem_image_path, *get_array_and_pixel_size(sem_image_path)
        )
        fib_info = TestImageInfo(
            fib_image_path, *get_array_and_pixel_size(fib_image_path)
        )
        logging.info("Loaded images")
        run_placement(
            i,
            milling_stages=milling_stages,
            sem_image=sem_info,  # type: ignore
            fib_image=fib_info,  # type: ignore
            plots_directory=plot_dir,
        )

In [9]:
def process_experiment(
    experiment_path: str | PathLike[str],
    plots_directory: str | PathLike[str],
    **protocol_kwargs: typing.Any,
) -> None:
    experiment_path = Path(experiment_path)
    plots_directory = Path(plots_directory)
    plots_directory.mkdir(exist_ok=True)
    experiment_yaml_path = experiment_path / "experiment.yaml"
    protocol_path = experiment_path / "protocol.yaml"
    assert protocol_path.is_file()

    with experiment_yaml_path.open() as f:
        experiment_dict = yaml.safe_load(f)

    experiment_positions = experiment_dict["positions"]

    for position in experiment_positions:
        lamella_name = position["petname"]
        expected_path = experiment_path / lamella_name
        if expected_path.is_dir():
            print(f"Starting pattern placement for {lamella_name}")
            with tempfile.TemporaryDirectory() as tmp_dir:
                protocol_path = setup_test_protocol(
                    {"milling": position["protocol"]},
                    temporary_directory=Path(tmp_dir),
                    model_path=model_path,
                    model_generation=model_generation,
                    **protocol_kwargs,
                )
                # Set up test microscope
                microscope, settings = utils.setup_session(
                    config_path=microscope_config_path, protocol_path=protocol_path
                )
            assert settings.protocol is not None

            milling_stages: list[FibsemMillingStage] = []
            rough_milling_stages = get_milling_stages(
                "mill_rough", settings.protocol["milling"]
            )
            for rough_stage in rough_milling_stages:
                if rough_stage.strategy.name == "BitmapAdaptivePolishing":
                    milling_stages.append(rough_stage)
                    break

            polishing_stages = get_milling_stages(
                "mill_polishing", settings.protocol["milling"]
            )
            for polishing_stage in polishing_stages[::-1]:
                if polishing_stage.strategy.name == "BitmapAdaptivePolishing":
                    milling_stages.append(polishing_stage)
                    break

            ap_subdirs = list(expected_path.glob("*adaptive_polish_*"))
            if ap_subdirs:
                for ap_dir in ap_subdirs:
                    print(
                        f"Starting pattern placement for {lamella_name}/{ap_dir.name}"
                    )
                    try:
                        fib_image_dir, sem_image_dir = get_fib_and_sem_directories(
                            ap_dir
                        )
                        plot_subdir = plots_directory / expected_path.name / ap_dir.name
                        plot_subdir.mkdir(exist_ok=True, parents=True)

                        process_lamella(
                            fib_path=fib_image_dir,
                            sem_path=sem_image_dir,
                            plot_dir=plot_subdir,
                            milling_stages=milling_stages,
                        )
                    except FileNotFoundError:
                        continue


In [ ]:
process_experiment(
    experiment_path,
    plots_directory=plots_directory,
    bitmap_gaussian_sigma=20,
    bitmap_erosion_px=40,
    apply_boundary_smoothing=True,
    lamella_gis_boundary_gaussian_sigma=15,
)

Starting pattern placement for 01-fit-troll
2025-10-24 22:02:43,432 — root — INFO — _setup_image_iterators:444 — SEM or FIB data path not configured in simulator settings, using random noise generation
2025-10-24 22:02:43,445 — root — INFO — connect_to_microscope:265 — Microscope client connected to DemoMicroscope with serial number 123456 and software version 0.1
2025-10-24 22:02:43,451 — root — INFO — setup_session:282 — Finished setup for session: openfibsem_2025-10-24-10-02-43PM
2025-10-24 22:02:44,096 — root — INFO — _get_plugin_strategies:56 — Loaded strategy 'AdaptivePolishing'
2025-10-24 22:02:44,099 — root — INFO — _get_plugin_strategies:56 — Loaded strategy 'BitmapAdaptivePolishing'
2025-10-24 22:02:44,102 — root — INFO — _get_plugin_strategies:56 — Loaded strategy 'AdaptivePolishing'
Starting pattern placement for 01-fit-troll/adaptive_polish_2025-05-13-10-36-45PM
2025-10-24 22:02:44,164 — root — INFO — process_lamella:20 — Loaded images
2025-10-24 22:02:44,169 — adaptive_po

c:\Users\tpr78264\code\adapive_polish_workspace\adaptive_polish\.venv\lib\site-packages\matplotlib_scalebar\scalebar.py:457: UserWarning: Drawing scalebar on axes with unequal aspect ratio; either call ax.set_aspect(1) or suppress the warning with rotation='horizontal-only'.
  warnings.warn(


2025-10-24 22:03:31,131 — adaptive_polish.processing.sem_segmentation — INFO — load_model:709 — Loading 1.4fpn generation model
2025-10-24 22:03:33,199 — root — INFO — run_placement:15 — Loaded model
2025-10-24 22:03:33,201 — root — INFO — run_placement:19 — Getting lamella info
2025-10-24 22:03:44,025 — root — INFO — run_placement:31 — Updating milling stage
2025-10-24 22:03:45,066 — adaptive_polish.strategy.bitmap — INFO — _update_milling_stage:95 — Created new TrenchBitmap for Adaptive Polishing
2025-10-24 22:03:45,070 — root — INFO — run_placement:39 — Creating stage plot
2025-10-24 22:03:58,610 — root — INFO — run_placement:46 — Creating multistage plot
Saving figure to "C:\Users\tpr78264\OneDrive - The Rosalind Franklin Institute\Documents\test data\adaptive milling\AM session 20251305\20250512_AM_all\20250513_AM_ON\AutoLamella-2025-05-13-16-35\bitmap_pattern_plots\01-fit-troll\adaptive_polish_2025-05-13-10-36-45PM\01-fit-troll_AP_img_000_SEM_1_1_bitmap_pattern_plot.tif"
2025-10-

KeyboardInterrupt: 